In [0]:
%python
from pyspark.sql.functions import col

bronze_df = spark.table("workspace.bronze.product_category")

silver_df = (bronze_df
    .withColumn("ProductCategoryID", col("ProductCategoryID").cast("int"))
    .withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))
)

# Duplicate check on the business key
dupe_count = (silver_df.groupBy("ProductCategoryID").count()
              .filter("count > 1").count())
print(f"Duplicate ProductCategoryIDs: {dupe_count}")

silver_df.write.mode("overwrite").saveAsTable("workspace.silver.product_category")

In [0]:
%python
from pyspark.sql.functions import col

bronze_df = spark.table("workspace.bronze.sales_territory")

silver_df = (bronze_df
    .withColumn("TerritoryID", col("TerritoryID").cast("int"))
    .withColumn("SalesYTD", col("SalesYTD").cast("decimal(18,4)"))
    .withColumn("SalesLastYear", col("SalesLastYear").cast("decimal(18,4)"))
    .withColumn("CostYTD", col("CostYTD").cast("decimal(18,4)"))
    .withColumn("CostLastYear", col("CostLastYear").cast("decimal(18,4)"))
    .withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))
)

dupe_count = (silver_df.groupBy("TerritoryID").count()
              .filter("count > 1").count())
print(f"Duplicate TerritoryIDs: {dupe_count}")

silver_df.write.mode("overwrite").saveAsTable("workspace.silver.sales_territory")

In [0]:
%python
from pyspark.sql.functions import col, when

bronze_df = spark.table("workspace.bronze.product")

silver_df = (bronze_df
    .withColumn("ProductID", col("ProductID").cast("int"))
    .withColumn("MakeFlag", when(col("MakeFlag") == "True", True)
                            .when(col("MakeFlag") == "False", False)
                            .otherwise(None))
    .withColumn("FinishedGoodsFlag", when(col("FinishedGoodsFlag") == "True", True)
                                      .when(col("FinishedGoodsFlag") == "False", False)
                                      .otherwise(None))
    .withColumn("SafetyStockLevel", col("SafetyStockLevel").cast("int"))
    .withColumn("ReorderPoint", col("ReorderPoint").cast("int"))
    .withColumn("StandardCost", col("StandardCost").cast("decimal(18,4)"))
    .withColumn("ListPrice", col("ListPrice").cast("decimal(18,4)"))
    .withColumn("Weight", col("Weight").cast("decimal(10,2)"))
    .withColumn("DaysToManufacture", col("DaysToManufacture").cast("int"))
    .withColumn("ProductSubcategoryID", col("ProductSubcategoryID").cast("int"))
    .withColumn("ProductModelID", col("ProductModelID").cast("int"))
    .withColumn("SellStartDate", col("SellStartDate").cast("timestamp"))
    .withColumn("SellEndDate", col("SellEndDate").cast("timestamp"))
    .withColumn("DiscontinuedDate", col("DiscontinuedDate").cast("timestamp"))
    .withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))
)

dupe_count = (silver_df.groupBy("ProductID").count()
              .filter("count > 1").count())
print(f"Duplicate ProductIDs: {dupe_count}")

# FK sanity check against product_category is skipped here — 
# ProductSubcategoryID would need the Subcategory table, which we didn't bring in.
# Worth noting as a documented simplification for your portfolio write-up.

silver_df.write.mode("overwrite").saveAsTable("workspace.silver.product")

In [0]:
%python
from pyspark.sql.functions import col, regexp_extract

bronze_df = spark.table("workspace.bronze.address")

silver_df = (bronze_df
    .withColumn("AddressID", col("AddressID").cast("int"))
    .withColumn("StateProvinceID", col("StateProvinceID").cast("int"))
    .withColumn("Longitude", regexp_extract(col("SpatialLocation"), r"POINT \(([-\d.]+) ([-\d.]+)\)", 1).cast("double"))
    .withColumn("Latitude", regexp_extract(col("SpatialLocation"), r"POINT \(([-\d.]+) ([-\d.]+)\)", 2).cast("double"))
    .withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))
)

dupe_count = (silver_df.groupBy("AddressID").count()
              .filter("count > 1").count())
print(f"Duplicate AddressIDs: {dupe_count}")

# Check the regex actually worked on a sample before trusting it at scale
silver_df.select("SpatialLocation", "Longitude", "Latitude").show(5, truncate=False)

silver_df.write.mode("overwrite").saveAsTable("workspace.silver.address")

In [0]:
%python
from pyspark.sql.functions import col

bronze_df = spark.table("workspace.bronze.customer")

silver_df = (bronze_df
    .withColumn("CustomerID", col("CustomerID").cast("int"))
    .withColumn("PersonID", col("PersonID").cast("int"))
    .withColumn("StoreID", col("StoreID").cast("int"))
    .withColumn("TerritoryID", col("TerritoryID").cast("int"))
    .withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))
)

dupe_count = (silver_df.groupBy("CustomerID").count()
              .filter("count > 1").count())
print(f"Duplicate CustomerIDs: {dupe_count}")

# FK check: any TerritoryID in customer that doesn't exist in sales_territory?
territories = spark.table("workspace.silver.sales_territory").select("TerritoryID")
orphans = (silver_df.select("TerritoryID").distinct()
           .join(territories, "TerritoryID", "left_anti"))
print(f"Orphaned TerritoryIDs: {orphans.count()}")

silver_df.write.mode("overwrite").saveAsTable("workspace.silver.customer")

In [0]:
%python
from pyspark.sql.functions import col, when

bronze_df = spark.table("workspace.bronze.sales_order_header")

silver_df = (bronze_df
    .withColumn("SalesOrderID", col("SalesOrderID").cast("int"))
    .withColumn("RevisionNumber", col("RevisionNumber").cast("int"))
    .withColumn("OrderDate", col("OrderDate").cast("timestamp"))
    .withColumn("DueDate", col("DueDate").cast("timestamp"))
    .withColumn("ShipDate", col("ShipDate").cast("timestamp"))
    .withColumn("Status", col("Status").cast("int"))
    .withColumn("OnlineOrderFlag", when(col("OnlineOrderFlag") == "True", True)
                                    .when(col("OnlineOrderFlag") == "False", False)
                                    .otherwise(None))
    .withColumn("CustomerID", col("CustomerID").cast("int"))
    .withColumn("SalesPersonID", col("SalesPersonID").cast("int"))
    .withColumn("TerritoryID", col("TerritoryID").cast("int"))
    .withColumn("BillToAddressID", col("BillToAddressID").cast("int"))
    .withColumn("ShipToAddressID", col("ShipToAddressID").cast("int"))
    .withColumn("ShipMethodID", col("ShipMethodID").cast("int"))
    .withColumn("CreditCardID", col("CreditCardID").cast("int"))
    .withColumn("CurrencyRateID", col("CurrencyRateID").cast("int"))
    .withColumn("SubTotal", col("SubTotal").cast("decimal(18,4)"))
    .withColumn("TaxAmt", col("TaxAmt").cast("decimal(18,4)"))
    .withColumn("Freight", col("Freight").cast("decimal(18,4)"))
    .withColumn("TotalDue", col("TotalDue").cast("decimal(18,4)"))
    .withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))
)

dupe_count = (silver_df.groupBy("SalesOrderID").count()
              .filter("count > 1").count())
print(f"Duplicate SalesOrderIDs: {dupe_count}")

# FK checks against dimensions we've already built
customers = spark.table("workspace.silver.customer").select("CustomerID")
territories = spark.table("workspace.silver.sales_territory").select("TerritoryID")
addresses = spark.table("workspace.silver.address").select("AddressID")

orphan_customers = silver_df.select("CustomerID").distinct().join(customers, "CustomerID", "left_anti").count()
orphan_territories = silver_df.select("TerritoryID").distinct().join(territories, "TerritoryID", "left_anti").count()
orphan_bill_addr = (silver_df.select(col("BillToAddressID").alias("AddressID")).distinct()
                     .join(addresses, "AddressID", "left_anti").count())
orphan_ship_addr = (silver_df.select(col("ShipToAddressID").alias("AddressID")).distinct()
                     .join(addresses, "AddressID", "left_anti").count())

print(f"Orphaned CustomerIDs: {orphan_customers}")
print(f"Orphaned TerritoryIDs: {orphan_territories}")
print(f"Orphaned BillToAddressIDs: {orphan_bill_addr}")
print(f"Orphaned ShipToAddressIDs: {orphan_ship_addr}")

silver_df.write.mode("overwrite").saveAsTable("workspace.silver.sales_order_header")

In [0]:
%python
from pyspark.sql.functions import col

bronze_df = spark.table("workspace.bronze.sales_order_detail")

silver_df = (bronze_df
    .withColumn("SalesOrderID", col("SalesOrderID").cast("int"))
    .withColumn("SalesOrderDetailID", col("SalesOrderDetailID").cast("int"))
    .withColumn("OrderQty", col("OrderQty").cast("int"))
    .withColumn("ProductID", col("ProductID").cast("int"))
    .withColumn("SpecialOfferID", col("SpecialOfferID").cast("int"))
    .withColumn("UnitPrice", col("UnitPrice").cast("decimal(18,4)"))
    .withColumn("UnitPriceDiscount", col("UnitPriceDiscount").cast("decimal(18,4)"))
    .withColumn("LineTotal", col("LineTotal").cast("decimal(18,4)"))
    .withColumn("ModifiedDate", col("ModifiedDate").cast("timestamp"))
)

# Real PK check
dupe_count = (silver_df.groupBy("SalesOrderDetailID").count()
              .filter("count > 1").count())
print(f"Duplicate SalesOrderDetailIDs: {dupe_count}")

# FK checks
orders = spark.table("workspace.silver.sales_order_header").select("SalesOrderID")
products = spark.table("workspace.silver.product").select("ProductID")

orphan_orders = silver_df.select("SalesOrderID").distinct().join(orders, "SalesOrderID", "left_anti").count()
orphan_products = silver_df.select("ProductID").distinct().join(products, "ProductID", "left_anti").count()

print(f"Orphaned SalesOrderIDs: {orphan_orders}")
print(f"Orphaned ProductIDs: {orphan_products}")

silver_df.write.mode("overwrite").saveAsTable("workspace.silver.sales_order_detail")